# 01 · Explore the processed Maharashtra table

Checks on `data/processed/cases_maharashtra.parquet` (10.06M cases, filed 2010–2018):
what each column means, null rates, whether codes join to names, sentinel values and date sanity.
Findings are copied into `docs/DATA.md`.

The machine has ~7 GB of RAM, so each section loads only the columns it needs.

In [1]:
import pandas as pd
from court_delay.config import PROCESSED, RAW

PQ = PROCESSED / "cases_maharashtra.parquet"
KEYS = RAW / "keys"
pd.set_option("display.max_rows", 60)

def load(*cols):
    return pd.read_parquet(PQ, columns=list(cols))

## 1. Column dictionary

| column | meaning | notes |
|---|---|---|
| `ddl_case_id` | DDL's unique case id | join key to `acts_sections` |
| `year` | filing year (2010–2018) | |
| `state_code`, `dist_code`, `court_no` | location codes | Maharashtra = state 1 |
| `judge_position` | designation of the presiding judge | 70 free-text values |
| `female_defendant`, `female_petitioner` | inferred gender of the parties | strings like `"0 male"`; sentinels −9998 unclear, −9999 missing |
| `female_adv_def`, `female_adv_pet` | inferred gender of the advocates | ints; same sentinels |
| `type_name` → `type_label` | case-type code → name | messy free text (see §7) |
| `purpose_name` | purpose of the latest hearing (code) | not joined yet; key is per year |
| `disp_name` → `disp_label` | disposition code → name | `"disposition var missing"` for pending |
| `date_of_filing`, `date_of_decision` | lifecycle dates (raw) | |
| `date_first_list`, `date_last_list`, `date_next_list` | first / last / next hearing | contain typo dates (2022, 2101) |
| `state_name`, `district_name` | joined labels | district joined on (state, dist) only, D-009 |
| `filing_date`, `decision_date` | cleaned dates | decision after the cutoff → censored |
| `event` | 1 = decided, 0 = pending (right-censored) | |
| `duration_days` | decision − filing, or cutoff − filing if censored | |
| `bad_dates` | decision before filing | |

## 2. Null rates

In [2]:
cols = ["judge_position", "type_label", "purpose_name", "disp_label", "district_name",
        "date_of_filing", "date_of_decision", "date_first_list", "date_last_list", "date_next_list"]
df = load(*cols)
print(f"{len(df):,} rows")
(df.isna().mean() * 100).round(2).rename("null %").to_frame()

10,059,428 rows


,null %
judge_position,0.00
type_label,0.00
purpose_name,2.23
disp_label,0.00
district_name,0.00
date_of_filing,0.00
date_of_decision,27.47
date_first_list,1.60
date_last_list,1.30
date_next_list,1.31


Only the decision date has many nulls (27.5%), and those are the pending cases. Hearing dates are
1–2% null and `purpose_name` 2.2%. Every row has a district, type and disposition label.

In [3]:
del df

## 3. Code → name joins

District, type and disposition labels were joined at ingest. Here we check that the two keys we
have not joined yet, **court** and **purpose**, would cover every code.

In [4]:
df = load("year", "dist_code", "court_no", "purpose_name")

ck = pd.read_csv(KEYS / "cases_court_key.csv")
ck = ck[ck.state_code == 1]
courts = df[["year", "dist_code", "court_no"]].drop_duplicates()
m = courts.merge(ck[["year", "dist_code", "court_no"]], how="left", indicator=True)
print("court (year, dist, court) join:", m._merge.value_counts().to_dict())
print("distinct courts:", courts[["dist_code", "court_no"]].drop_duplicates().shape[0])
print("courts whose name changes across years:",
      ck.groupby(["dist_code", "court_no"]).court_name.nunique().gt(1).sum())

pk = pd.read_csv(KEYS / "purpose_name_key.csv")
pp = df[["year", "purpose_name"]].dropna().drop_duplicates()
m = pp.merge(pk[["year", "purpose_name"]], how="left", indicator=True)
print("purpose (year, code) join:", m._merge.value_counts().to_dict())
print("distinct purpose codes:", df.purpose_name.nunique())

court (year, dist, court) join: {'both': 5634, 'left_only': 0, 'right_only': 0}
distinct courts: 633
courts whose name changes across years: 11


purpose (year, code) join: {'both': 9296, 'left_only': 0, 'right_only': 0}
distinct purpose codes: 5550


Both keys join fully on (year, code). Unlike the district key, the court key **is** per year.
Purpose codes are year-specific: about 5.5k codes, so they must be joined on year and normalised
before they are used as a feature.

In [5]:
del df

## 4. Judge position and gender flags

In [6]:
df = load("judge_position", "female_defendant", "female_petitioner", "female_adv_def", "female_adv_pet")
print("judge_position:", df.judge_position.nunique(), "values")
display(df.judge_position.value_counts().head(12).to_frame())
for c in ["female_defendant", "female_petitioner", "female_adv_def", "female_adv_pet"]:
    display((df[c].value_counts(normalize=True) * 100).round(1).rename(f"{c} %").to_frame())

judge_position: 70 values


,count
judge_position,
criminal cases,1451236
judicial magistrate court,1317213
district and sessions court,1258599
chief judicial magistrate,1236880
civil judge junior division,820409
civil court,685188
civil judge senior division,625420
metropolitan magistrate court,431430
civil judge,271409


,female_defendant %
female_defendant,
0 male,68.0
-9998 unclear,22.6
1 female,9.2
-9999 missing name,0.2


,female_petitioner %
female_petitioner,
-9998 unclear,48.4
0 male,37.7
1 female,13.9
-9999 missing name,0.0


,female_adv_def %
female_adv_def,
-9999,75.8
0,19.4
-9998,3.0
1,1.7


,female_adv_pet %
female_adv_pet,
0,55.2
-9999,20.3
-9998,19.8
1,4.7


The gender flags use sentinels: **−9998 = unclear** and **−9999 = missing name**. They are not NaN.
Petitioner gender is unclear for about 48% of cases, which are often the State or a company.
Advocate gender is missing for 76% (defendant side). Treat these as categories, never as numbers.

In [7]:
del df

## 5. Dispositions vs decision dates

In [8]:
df = load("disp_label", "date_of_decision", "event")
no_date = df.date_of_decision.isna()
missing = df.disp_label.eq("disposition var missing")
print(pd.crosstab(no_date.rename("no decision date"), missing.rename("disp missing"), margins=True))
print("\nNo decision date but a real disposition:")
display(df.loc[no_date & ~missing, "disp_label"].value_counts().head(10).to_frame())
print("Decided cases by disposition:")
display((df.loc[df.event == 1, "disp_label"].value_counts(normalize=True) * 100).round(1).head(15).to_frame())

disp missing        False     True       All
no decision date                            
False             7262675    33329   7296004
True                 1717  2761707   2763424
All               7264392  2795036  10059428

No decision date but a real disposition:


,count
disp_label,
disposed-otherwise,799
transferred,253
dismissed,153
judgement,128
acquitted,58
referred to lok adalat,56
other,47
otherwise,37
compromise,34


Decided cases by disposition:


,proportion
disp_label,
disposed-otherwise,24.9
judgement,15.5
acquitted,8.0
referred to lok adalat,7.5
dismissed,6.3
allowed,5.2
258 crpc,3.9
other,3.6
withdrawn,3.5


- **Pending cases** are those with no decision date, and almost all of them have
  `disposition var missing`.
- Earlier, "1,022 cases with a disposition but no date" counted the `var missing` label as a
  disposition. With that label excluded, only about **1,700 of 10M (0.02%)** carry a real
  disposition but have no date. Leaving them censored has a negligible effect.
- About 33k decided cases have no disposition label. They are still decided.
- **Transferred** (~236k) and **referred to Lok Adalat** (~546k) end the case *in this court*
  but do not resolve the dispute. Decide in Week 3 whether these are events, censoring, or
  competing risks.

In [9]:
del df

## 6. Dates and durations

In [10]:
df = load("filing_date", "date_first_list", "duration_days", "bad_dates", "event")
print("bad_dates rows:", df.bad_dates.sum())
display(df.loc[df.bad_dates, "duration_days"].describe().to_frame("duration (bad rows)"))
print("negative durations outside bad_dates:", ((df.duration_days < 0) & ~df.bad_dates).sum())
print("zero-day durations:", (df.duration_days == 0).sum(),
      f"({(df.duration_days == 0).mean():.1%})")
print("zero-day share among decided:", f"{(df.loc[df.event == 1, 'duration_days'] == 0).mean():.1%}")
print("first hearing before filing:", (df.date_first_list < df.filing_date).sum())

bad_dates rows: 52370


,duration (bad rows)
count,52370.000000
mean,-25.128241
std,173.652828
min,-5526.000000
25%,-15.000000
50%,-5.000000
75%,-2.000000
max,-1.000000


negative durations outside bad_dates: 0
zero-day durations: 668934 (6.6%)


zero-day share among decided: 9.2%
first hearing before filing: 50007


- All 52k `bad_dates` rows have **negative** durations, so they must be filtered out before any
  survival fit. No other rows are negative.
- About 669k cases (6.6%) are decided **on the filing day**, usually plea or bail matters.
  Kaplan-Meier handles zeros, but parametric and AFT models need duration > 0, so use 0.5 days.
- 50k rows list a first hearing before filing, which is a typo. Ignore `date_first_list` as a feature
  unless it is cleaned.

In [11]:
del df

## 7. Case types are messy

In [12]:
df = load("type_label")
vc = df.type_label.value_counts()
print("distinct type labels:", len(vc))
cum = vc.cumsum() / vc.sum()
for n in (10, 25, 50, 100):
    print(f"top {n:>3} labels cover {cum.iloc[n-1]:.1%}")
vc.head(30).to_frame()

distinct type labels: 1160
top  10 labels cover 70.2%
top  25 labels cover 82.5%
top  50 labels cover 90.8%
top 100 labels cover 96.0%


,count
type_label,
s.c.c.,3040273
cri.m.a.,924419
r.c.c.,915242
r.c.s.,723468
cri.bail appln.,366066
civil m.a.,340449
other misc.cri.appln,212115
reg dkst,187915
marriage petn.,179683


There are 1,160 labels for what are really a few dozen case types, with spelling variants such as
`ss cases`, `ss casess` and `s s`. **Week 2 task:** map the labels to a small `case_category`
(civil suit, criminal summary, bail, MACT, matrimonial and so on) plus a civil/criminal flag.

In [13]:
del df

## Summary

| check | result |
|---|---|
| Labels present | district, type and disposition labels are 100% joined |
| Court key | joins 100% on (year, dist, court); 633 courts |
| Purpose key | joins 100% on (year, code); ~5.5k codes, needs normalising |
| Gender flags | −9998 / −9999 sentinels; treat as categories |
| Disposition without a date | ~1,700 real cases (0.02%); kept as censored |
| Transfer / Lok Adalat | ~780k "decisions" that don't resolve the dispute; decide in Week 3 |
| Bad dates | 52k negative durations; drop before modelling |
| Zero-day cases | 6.6%; floor at 0.5 days for parametric models |
| Case types | 1,160 messy labels; map to categories in Week 2 |